# Tutorial 11: Unsupervised Learning and Clustering
## Find structure without pretending that discovered groups are known truth

**Course:** IE 1171  
**File used:** `creditdata.csv`  
**Level 1:** Required core—K-means representation, selection, profiling, and stability  
**Level 2:** Optional deep dive—hierarchical clustering and linkage sensitivity

---

Supervised learning predicts a known response. Unsupervised learning receives no response label and searches for structure in the predictors. This tutorial uses the same credit-card behavior dataset and clustering readings as the earlier course notebook. Claude will help coordinate the code, but the human must define similarity, choose the representation, test stability, and decide whether the discovered groups have a defensible meaning.


# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Explain unsupervised learning and similarity**
   - Distinguish predictors from response labels.
   - Explain how distance, units, scaling, missing values, and skew define what “similar” means.
   - Explain why unsupervised learning has no ordinary classification accuracy.
2. **Build and interpret a K-means workflow**
   - Connect centroids, within-cluster variation, local optima, and random initialization.
   - compare candidate values of $K$ with inertia, silhouette score, profiles, and purpose.
   - Describe clusters in original units without treating cluster numbers as natural categories.
3. **Evaluate whether discovered groups are credible**
   - Check sensitivity to initialization and feature scaling.
   - Read a dendrogram and compare hierarchical linkage choices.
   - Examine how customer segmentation can create proxy groups, unequal treatment, and feedback loops.


# Pólya’s Four-Step Problem-Solving Cycle

> **Backbone for this tutorial:** George Pólya’s four steps organize the work from problem framing through verification. The steps are a **cycle**, not a one-way checklist: if later evidence exposes a bad assumption, return to the earlier step that needs revision.

| Marker | Pólya step | Guiding question | In this tutorial |
|---|---|---|---|
| **🔵 🧭** | **Understand the Problem** | What is the real problem, what is known, and what constraints define success? | Define the unsupervised question and what similarity should mean before creating clusters. |
| **🟣 🗺️** | **Devise a Plan** | What sequence of actions and checks should connect the current state to the goal? | Choose features, scaling, candidate values of K, and diagnostics before fitting. |
| **🟠 🛠️** | **Carry Out the Plan** | Can the plan be executed in small, observable steps and checked as it runs? | Fit and profile candidate clusters using the planned representation and diagnostics. |
| **🟢 🔎** | **Look Back** | Does the result answer the original problem, and what should be revised or generalized? | Stress-test stability and scale sensitivity, and avoid treating discovered clusters as ground truth. |

The colored markers reappear at the points where each step becomes the main focus. **Human Checks support the cycle, but they are not a universal checklist:** meaningful verification depends on domain knowledge, the data-generating process, and the consequences of being wrong.


# <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="44" style="vertical-align:middle; margin-right:10px;"> Assigned Reading

**James et al., _An Introduction to Statistical Learning with Applications in Python_ (ISLP)**

- Section 12.1: The Challenge of Unsupervised Learning
- Section 12.4: Clustering Methods
  - 12.4.1: K-Means Clustering
  - 12.4.2: Hierarchical Clustering
  - 12.4.3: Practical Issues in Clustering

The section range directly covers the K-means objective, local optima, hierarchical clustering, linkage, scaling, and practical interpretation used here.

The social-good section later in the notebook draws on earlier course discussions of recommendation, civic data, fairness, and stakeholder engagement, but **those materials are not additional assigned reading here**.

Level 2 uses the same `creditdata.csv` file and the same ISLP clustering chapter. No new dataset or reading is introduced.


# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

| Part | Purpose |
|---|---|
| **1. Define the problem** | Separate unsupervised structure from prediction. |
| **2. Build the representation** | Audit, impute, and standardize customer behavior variables. |
| **3. Compare values of K** | Use inertia and silhouette as evidence, not automatic answers. |
| **4. Fit and profile** | Describe groups in original units and visualize them with PCA. |
| **5. Check stability** | Repeat initialization and test sensitivity to scale. |
| **6. Level 2** | Compare hierarchical linkage choices and dendrogram cuts. |

## Human–AI rule

> Claude may calculate clusters. The human must define similarity, decide whether the solution is stable and useful, and reject labels that turn an exploratory grouping into a claim about people.


## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| **🔵 🧭  🟣 🗺️  🟠 🛠️  🟢 🔎** | **Pólya Backbone** | Treat the four colored checkpoints as the main problem-solving cycle; return to an earlier step when new evidence requires revision. |
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook's normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect equations, assumptions, and concepts to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the details an AI collaborator can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run the response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check — domain expertise required** | Use the provided questions, then add a domain-specific check. The notebook cannot supply a complete checklist for every application. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |

> **Important — Human Checks are not a complete checklist.** The notebook can suggest generic verification questions, but deciding what *must* be checked depends on knowledge of the domain, the data-generating process, and the consequences of an error. If you do not have that expertise, involve someone who does. Every Human Check asks you to add your own domain-specific question.

# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: Clustering Depends on Representation

Suppose $X\in\mathbb{R}^{n\times p}$ contains $n$ customers and $p$ behavior variables, but no response $Y$. Clustering seeks groups using only $X$. The result is not discovered independently of human choices; it depends on:

- which rows and variables enter $X$;
- missing-value treatment;
- transformations and scaling;
- the distance measure;
- the clustering algorithm and its tuning choices.

K-means partitions the observations into $K$ non-overlapping clusters $C_1,\ldots,C_K$ by minimizing within-cluster sum of squares:

$$
\min_{C_1,\ldots,C_K}
\sum_{k=1}^{K}\sum_{i\in C_k}\lVert x_i-\mu_k\rVert_2^2,
$$

where $\mu_k$ is the centroid of cluster $k$. The standard algorithm alternates between assigning observations to the nearest centroid and recomputing centroids. Each step improves or preserves the objective, but the final answer can be a local optimum. Multiple initializations are therefore necessary.

Euclidean distance is scale-sensitive. If balance ranges into thousands while a frequency lies between 0 and 1, unscaled balance can dominate the grouping. Standardization changes the question to similarity in standard-deviation units:

$$
z_{ij}=\frac{x_{ij}-\bar x_j}{s_j}.
$$

Inertia decreases whenever $K$ increases, so it cannot choose $K$ by itself. Silhouette score compares within-cluster cohesion with separation from the nearest alternative cluster and ranges roughly from -1 to 1. Neither score proves that a cluster is real, causal, fair, or useful.

Hierarchical clustering instead builds a tree of fusions. A dendrogram records which observations or clusters merge and at what dissimilarity. Linkage defines distance between clusters, so different linkage rules can produce different trees.

### Questions you should be ready to answer

- Why is there no response accuracy in this tutorial?
- How does scaling define similarity?
- Why can two K-means runs disagree?
- What evidence should accompany a choice of $K$?
- Why are cluster labels descriptive rather than ground-truth identities?


## 🔵 🧭 Pólya Step 1 — Understand the Problem

**Backbone checkpoint.** State the real goal, evidence, constraints, and what would count as success before asking an AI system to solve anything.

**In this tutorial:** Define the unsupervised question and what similarity should mean before creating clusters.

# Level 1 — Required Core

# Part 1: Define the Unsupervised Question

The file `creditdata.csv` describes credit-card behavior for roughly 9,000 customers across variables such as:

- balance and balance frequency;
- purchases, one-off purchases, and installment purchases;
- cash advances and transaction counts;
- credit limit and payments;
- minimum payments and full-payment proportion;
- account tenure.

`CUST_ID` identifies a customer but does not measure behavior. It must not be used in a distance calculation.

The goal is exploratory:

> Can customers be represented as a small number of behavior groups that are stable enough to summarize, while keeping the limitations of segmentation visible?

There is no supplied “correct cluster” column.


## 🟣 🗺️ Pólya Step 2 — Devise a Plan

**Backbone checkpoint.** Decide the sequence of actions and checks before the main execution. Make assumptions, evaluation rules, and stopping conditions visible so they can be challenged.

**In this tutorial:** Choose features, scaling, candidate values of K, and diagnostics before fitting.

## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Define Similarity Before Coding

1. What is one observational unit?
2. Why is `CUST_ID` excluded?
3. Which variables represent amounts, frequencies, counts, proportions, and duration?
4. Which variables are likely to be highly skewed?
5. What does it mean for two customers to be close?
6. Should every variable receive equal standardized influence?
7. Which missing values could have substantive meaning?
8. What evidence would make a cluster solution useful?
9. What evidence would make you reject it?
10. Who could be harmed if clusters control offers or credit access?


## 🟠 🛠️ Pólya Step 3 — Carry Out the Plan

**Backbone checkpoint.** Execute in small, observable steps. Read generated code or actions, stay within scope, and compare outputs with the behavior you predicted.

**In this tutorial:** Fit and profile candidate clusters using the planned representation and diagnostics.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 1: Load and Audit the Credit Behavior Data

        ```text
        Act as a careful Python tutor.

Write one Jupyter Notebook code cell that:

1. imports pathlib, NumPy, pandas, matplotlib, seaborn,
   sklearn SimpleImputer, StandardScaler, KMeans, PCA,
   silhouette_score, adjusted_rand_score, scipy linkage,
   dendrogram, fcluster, and pdist;
2. loads creditdata.csv;
3. confirms that CUST_ID exists;
4. prints the shape, first five rows, data types, duplicate-row count,
   duplicated-ID count, and missing-value counts;
5. lists numeric and nonnumeric columns separately;
6. displays descriptive statistics for numeric behavior variables;
7. does not impute, scale, or cluster.

Name the dataframe credit.
Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 1


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import pdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

credit_path = Path("creditdata.csv")
assert credit_path.exists(), f"File not found: {credit_path.resolve()}"

credit = pd.read_csv(credit_path)
assert "CUST_ID" in credit.columns, "Required identifier CUST_ID is missing."

print("Shape:", credit.shape)
display(credit.head())
print("\nData types:")
display(credit.dtypes.to_frame("dtype"))
print("\nExact duplicate rows:", int(credit.duplicated().sum()))
print("Duplicated customer IDs:", int(credit["CUST_ID"].duplicated().sum()))
print("\nMissing values:")
display(
    credit.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

numeric_columns = credit.select_dtypes(include=np.number).columns.tolist()
nonnumeric_columns = [
    column for column in credit.columns if column not in numeric_columns
]
print("\nNumeric columns:", numeric_columns)
print("\nNonnumeric columns:", nonnumeric_columns)
display(credit[numeric_columns].describe().T)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Is one row one customer?
- Is `CUST_ID` unique, and does it contain any behavior information?
- Which variables have missing values?
- Which variables have maxima far above their medians?
- Do any columns appear duplicated, constant, or incorrectly typed?
- Does the data dictionary support every interpretation you plan to make?

# Part 2: Build the Feature Representation

The earlier course notebook visualized three variables and normalized them to a 0–1 range. This revision keeps the same dataset but uses all numeric behavior variables so the clusters represent the broader customer profile.

The required representation:

1. excludes `CUST_ID`;
2. imputes each numeric variable with its median;
3. standardizes each variable to mean approximately zero and standard deviation approximately one.

Median imputation and equal standardized weight are modeling choices, not neutral facts. A domain expert may prefer a missingness indicator, a log transformation for heavy tails, or a smaller feature set that matches the actual decision.


## <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="36" style="vertical-align:middle; margin-right:9px;"> Without Claude: What Must Stay Aligned?

A clustering workflow must preserve:

- the exact order of feature columns;
- the fitted median for each feature;
- the fitted mean and scale for each feature;
- the connection between each transformed row and its customer ID;
- the original units needed for interpretation.

If any of these become misaligned, the code can still run while the profiles become wrong.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 2: Impute and Standardize the Behavior Variables

        ```text
        Using credit:

1. create cluster_features from numeric columns only and exclude CUST_ID;
2. assert that at least two features remain;
3. create X_raw with those features;
4. median-impute X_raw with SimpleImputer;
5. standardize the imputed matrix with StandardScaler;
6. name the arrays X_imputed and X_scaled;
7. create an imputed DataFrame in original units with the original index;
8. display each feature's imputation value, original median, transformed mean,
   and transformed standard deviation;
9. verify that rows and feature columns remain aligned;
10. fit no clustering model.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 2


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2


In [ ]:
cluster_features = [
    column
    for column in credit.select_dtypes(include=np.number).columns
    if column != "CUST_ID"
]
assert len(cluster_features) >= 2

X_raw = credit[cluster_features].copy()
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X_raw)
X_scaled = scaler.fit_transform(X_imputed)

X_imputed_df = pd.DataFrame(
    X_imputed, columns=cluster_features, index=credit.index
)
representation_audit = pd.DataFrame(
    {
        "feature": cluster_features,
        "imputation_value": imputer.statistics_,
        "original_median": X_raw.median().to_numpy(),
        "scaled_mean": X_scaled.mean(axis=0),
        "scaled_std": X_scaled.std(axis=0, ddof=0),
    }
)
display(representation_audit)

assert X_scaled.shape == (len(credit), len(cluster_features))
assert list(X_imputed_df.columns) == cluster_features
assert X_imputed_df.index.equals(credit.index)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


1. Confirm that `CUST_ID` is absent from `cluster_features`.
2. Which medians replaced missing values?
3. Are transformed means near zero and standard deviations near one?
4. Does standardization make every variable equally important to the application?
5. Which heavy-tailed variables might still dominate through outliers?
6. What alternative representation would you test as a sensitivity analysis?

# Part 3: Compare Candidate Values of K

K-means requires the number of clusters in advance. Two common diagnostics are:

- **inertia:** total within-cluster squared distance; lower is better but always decreases as $K$ grows;
- **silhouette score:** compares cohesion with separation; higher is better for the chosen distance.

The elbow and the highest silhouette score are evidence, not commands. A defensible $K$ also needs stable profiles, adequate group sizes, and a purpose that can be explained without inventing identities.


## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: What Would Count as a Good K?

1. Why is $K=1$ excluded from silhouette score?
2. Why does inertia always favor more clusters?
3. What would a silhouette score near zero suggest?
4. Is the mathematically highest score automatically the most useful $K$?
5. What minimum cluster size would be interpretable?
6. How would you detect a cluster created by only a few outliers?
7. Why should the final decision be documented before naming the groups?


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 3: Evaluate K From 2 Through 8

        ```text
        Using X_scaled:

1. fit KMeans for K=2 through K=8 with random_state=1099 and n_init=25;
2. record inertia and silhouette score for each K;
3. if there are more than 3000 rows, calculate silhouette on a reproducible
   3000-row sample; otherwise use all rows;
4. create one table and side-by-side inertia and silhouette plots;
5. identify the K with the highest silhouette score as a candidate,
   not an automatic truth;
6. print a reminder to inspect stability, cluster sizes, and profiles;
7. save the table as k_evidence and the candidate as candidate_k.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 3


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3


In [ ]:
k_rows = []
silhouette_kwargs = (
    {"sample_size": 3000, "random_state": 1099}
    if len(X_scaled) > 3000
    else {}
)

for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=25, random_state=1099)
    labels = model.fit_predict(X_scaled)
    k_rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(
                X_scaled, labels, **silhouette_kwargs
            ),
            "smallest_cluster": int(
                pd.Series(labels).value_counts().min()
            ),
        }
    )

k_evidence = pd.DataFrame(k_rows)
candidate_k = int(
    k_evidence.loc[k_evidence["silhouette"].idxmax(), "k"]
)
display(k_evidence)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(k_evidence["k"], k_evidence["inertia"], marker="o")
axes[0].set(xlabel="K", ylabel="Inertia", title="Within-cluster variation")
axes[1].plot(
    k_evidence["k"], k_evidence["silhouette"], marker="o", color="#F58518"
)
axes[1].set(xlabel="K", ylabel="Silhouette score", title="Cohesion and separation")
plt.tight_layout()
plt.show()

print("Highest-silhouette candidate K:", candidate_k)
print(
    "Treat this as a candidate. Inspect stability, cluster sizes, "
    "profiles, and purpose before choosing K."
)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Where does inertia begin to flatten?
- Which $K$ has the highest silhouette score?
- Does that solution contain a very small group?
- Are neighboring values of $K$ nearly tied?
- Would the same $K$ be useful for description and for an operational decision?
- What evidence remains missing before choosing?

# Part 4: Fit, Visualize, and Profile the Candidate Solution

Cluster numbers are arbitrary. “Cluster 0” is not naturally lower, earlier, or better than “Cluster 1.” Interpret a group through transparent summaries in the original units.

PCA can project the standardized feature matrix to two dimensions for visualization. Tutorial 10 explains what that projection preserves and loses. The K-means model is still fit on the full standardized representation; the PCA plot is only a view.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 4: Fit and Profile the Candidate Clusters

        ```text
        Using candidate_k, X_scaled, X_imputed_df, and credit:

1. set selected_k = candidate_k and clearly mark where a justified override can be made;
2. fit KMeans with n_init=50 and random_state=1099;
3. save labels as cluster_label in a copy named credit_clustered;
4. show cluster counts and proportions;
5. show median feature profiles in original units;
6. show mean standardized profiles in a heatmap;
7. fit a two-component PCA only for visualization;
8. plot the PCA scores colored by cluster label and report both PVEs;
9. label the plot as a projection rather than proof of natural groups;
10. do not assign personality names to clusters.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 4


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4


In [ ]:
# Replace this only if you have documented evidence for another K.
selected_k = candidate_k

kmeans_final = KMeans(
    n_clusters=selected_k, n_init=50, random_state=1099
)
final_labels = kmeans_final.fit_predict(X_scaled)

credit_clustered = credit.copy()
credit_clustered["cluster_label"] = final_labels
X_profile = X_imputed_df.copy()
X_profile["cluster_label"] = final_labels

cluster_sizes = (
    credit_clustered["cluster_label"]
    .value_counts()
    .sort_index()
    .to_frame("customers")
)
cluster_sizes["proportion"] = cluster_sizes["customers"] / len(
    credit_clustered
)
display(cluster_sizes)

median_profiles = X_profile.groupby("cluster_label")[cluster_features].median()
display(median_profiles)

standardized_profiles = (
    pd.DataFrame(X_scaled, columns=cluster_features)
    .assign(cluster_label=final_labels)
    .groupby("cluster_label")
    .mean()
)
plt.figure(figsize=(12, max(3, 0.55 * selected_k + 1)))
sns.heatmap(
    standardized_profiles,
    cmap="vlag",
    center=0,
    cbar_kws={"label": "Mean standardized value"},
)
plt.title("Candidate cluster profiles in standardized units")
plt.xlabel("Behavior feature")
plt.ylabel("Arbitrary cluster label")
plt.tight_layout()
plt.show()

pca_view = PCA(n_components=2)
pca_coordinates = pca_view.fit_transform(X_scaled)
pca_plot = pd.DataFrame(
    {
        "PC1": pca_coordinates[:, 0],
        "PC2": pca_coordinates[:, 1],
        "cluster_label": final_labels.astype(str),
    }
)
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=pca_plot,
    x="PC1",
    y="PC2",
    hue="cluster_label",
    alpha=0.45,
    s=25,
    palette="tab10",
)
plt.title("Two-component projection of full-space K-means labels")
plt.tight_layout()
plt.show()
print("PCA visualization PVE:", pca_view.explained_variance_ratio_)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


1. Are any groups too small to summarize reliably?
2. Which original variables distinguish each group most strongly?
3. Are profiles driven by a few extreme observations?
4. Do mean and median profiles tell the same story?
5. How much variance does the two-dimensional view omit?
6. Could apparent overlap in the plot hide separation in later dimensions?
7. Are your descriptions behaviors rather than judgments about people?

# Part 5: Check Stability and Scale Sensitivity

A useful cluster solution should not disappear after a harmless change in random initialization. Adjusted Rand index (ARI) compares two partitions while ignoring arbitrary label numbers:

- ARI near 1: highly similar partitions;
- ARI near 0: agreement similar to chance;
- negative ARI: less agreement than expected by chance.

Scale sensitivity asks a different question. If standardized and unscaled results disagree, neither is automatically wrong. They encode different definitions of similarity.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 5: Stress-Test the Candidate Clusters

        ```text
        Using selected_k, X_scaled, X_imputed, and final_labels:

1. fit KMeans with n_init=1 for random seeds 0 through 19;
2. calculate adjusted Rand index between each result and final_labels;
3. display and plot the ARI distribution;
4. fit the same K to unscaled median-imputed data with n_init=50;
5. calculate ARI between standardized and unscaled partitions;
6. compare cluster-size proportions for both representations;
7. do not describe disagreement as a software error;
8. print one interpretation question about which similarity definition fits the purpose.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 5


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5


In [ ]:
stability_rows = []
for seed in range(20):
    labels_seed = KMeans(
        n_clusters=selected_k, n_init=1, random_state=seed
    ).fit_predict(X_scaled)
    stability_rows.append(
        {
            "seed": seed,
            "ari_vs_reference": adjusted_rand_score(
                final_labels, labels_seed
            ),
        }
    )

stability = pd.DataFrame(stability_rows)
display(stability.describe())
plt.figure(figsize=(7, 4))
sns.stripplot(data=stability, x="ari_vs_reference", color="#4C78A8")
plt.xlim(-0.05, 1.05)
plt.title("Initialization stability against the reference solution")
plt.xlabel("Adjusted Rand index")
plt.tight_layout()
plt.show()

unscaled_labels = KMeans(
    n_clusters=selected_k, n_init=50, random_state=1099
).fit_predict(X_imputed)
scale_ari = adjusted_rand_score(final_labels, unscaled_labels)
print("ARI: standardized versus unscaled partition:", f"{scale_ari:.3f}")

standardized_size_ranks = (
    pd.Series(final_labels)
    .value_counts(normalize=True)
    .sort_values(ascending=False)
    .reset_index(drop=True)
)
unscaled_size_ranks = (
    pd.Series(unscaled_labels)
    .value_counts(normalize=True)
    .sort_values(ascending=False)
    .reset_index(drop=True)
)
scale_sizes = pd.DataFrame(
    {
        "size_rank": np.arange(1, selected_k + 1),
        "standardized_proportion": standardized_size_ranks,
        "unscaled_proportion": unscaled_size_ranks,
    }
)
display(scale_sizes)
print(
    "Interpretation question: should similarity be measured in original "
    "units or in standard-deviation units for this purpose?"
)


## 🟢 🔎 Pólya Step 4 — Look Back

**Backbone checkpoint.** Do not stop at “it ran.” Ask whether the result answers the original problem, what evidence supports it, what failed, and what should change. Domain expertise matters here because a generic checklist cannot know every real-world failure mode.

**In this tutorial:** Stress-test stability and scale sensitivity, and avoid treating discovered clusters as ground truth.

### <img src="tutorial-icons/look_back.png" alt="Look Back" width="30" style="vertical-align:middle; margin-right:8px;"> Level 1 Look Back

1. Is the solution stable across initializations?
2. How much does scaling change the partition?
3. Which representation matches the stated purpose?
4. Is the chosen $K$ supported by more than one diagnostic?
5. Are the profiles distinct in original units?
6. Which claims remain exploratory?
7. What new data or external validation would strengthen the interpretation?


# AI for Social Good: Clusters Are Analytical Choices, Not Natural Types of People

Clustering can help public-interest organizations summarize service needs, detect infrastructure patterns, organize public-health data, or explore civic datasets. But segmentation can become unequal treatment when a cluster label is later used for prices, credit, policing, benefits, advertising, or access.

The earlier course emphasized stakeholder engagement and public accountability. Those ideas are especially important here because there is no response variable telling us that a discovered group is “correct.” The grouping depends on the features, distance measure, scaling, number of clusters, and the data that were collected.

Before using a cluster in a decision process:

- ask whether the features encode protected traits or socioeconomic proxies;
- compare missingness and outlier treatment across affected groups;
- test how stable clusters are under reasonable changes to scaling and initialization;
- avoid labels that turn measured behavior into identity;
- ask whether communities recognize the interpretation being assigned to them;
- avoid feedback loops in which a cluster receives different treatment and then looks even more different in future data;
- provide a way to challenge consequential segmentation.

> **Social-good principle:** A cluster is a modeler-created summary of similarity. It should not become a fixed identity or entitlement rule without strong domain and stakeholder justification.


# Tutorial 11 Conclusion

You used Claude to audit credit behavior data, create a documented feature representation, compare values of $K$, fit and profile K-means clusters, visualize the solution with PCA, and test sensitivity to initialization and scaling.

The main lesson is that unsupervised learning can reveal useful structure without a response label, but the structure depends on human choices about variables, units, distance, algorithm, and purpose.


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Final Reflection

1. How does unsupervised learning differ from supervised learning?
2. Why is `CUST_ID` excluded?
3. What does standardization change?
4. What objective does K-means minimize?
5. Why are multiple initializations necessary?
6. What do inertia and silhouette score reveal?
7. Why can neither metric prove the groups are real?
8. Why are cluster numbers arbitrary?
9. What does ARI measure?
10. Which interpretation could not be delegated to Claude?


# <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 — Optional Deep Dive

> **Challenge ahead:** Complete Level 1 first. This route uses the same credit data and ISLP Section 12.4 to compare K-means with hierarchical clustering.

# Part 6: Hierarchical Clustering and Dendrograms

Agglomerative hierarchical clustering begins with each observation in its own cluster, then repeatedly fuses the closest pair. The dendrogram shows the sequence and height of these fusions.

Linkage defines distance between clusters:

- **single:** closest pair of observations;
- **complete:** farthest pair;
- **average:** average pairwise distance;
- **Ward:** increase in within-cluster squared variation.

A dendrogram's horizontal order is mainly a layout. The fusion height carries the dissimilarity information.


## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Predict Linkage Behavior

1. Which linkage is vulnerable to chaining?
2. Which linkage tends to prefer compact groups?
3. Why is Ward naturally paired with Euclidean geometry?
4. What does a high fusion represent?
5. Why do we sample customers before drawing the dendrogram?
6. Does a horizontal cut reveal the one correct number of clusters?
7. What information is lost when only 80 customers are displayed?


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 6: Build and Read a Ward Dendrogram

        ```text
        Using X_scaled and selected_k:

1. select 80 row positions with np.random.default_rng(1099) without replacement;
2. fit Ward hierarchical clustering to those standardized rows;
3. draw a readable dendrogram with leaf labels based on row positions;
4. draw a horizontal line that produces selected_k groups;
5. cut the tree into selected_k labels with fcluster;
6. show hierarchical group sizes;
7. compare the hierarchical labels with the K-means labels for the same rows using ARI;
8. explain in a printed note that the sampled dendrogram is a teaching view,
   not a full-data validation.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 6


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 6


In [ ]:
rng = np.random.default_rng(1099)
sample_positions = np.sort(
    rng.choice(len(X_scaled), size=min(80, len(X_scaled)), replace=False)
)
X_hierarchical = X_scaled[sample_positions]

ward_tree = linkage(X_hierarchical, method="ward", metric="euclidean")
ward_labels = fcluster(
    ward_tree, t=selected_k, criterion="maxclust"
)

plt.figure(figsize=(14, 6))
dendrogram(
    ward_tree,
    labels=[str(position) for position in sample_positions],
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=None,
)
cut_height = ward_tree[-(selected_k - 1), 2] if selected_k > 1 else 0
plt.axhline(cut_height, color="#E45756", linestyle="--")
plt.title("Ward dendrogram for a reproducible customer sample")
plt.xlabel("Sampled row position")
plt.ylabel("Fusion height")
plt.tight_layout()
plt.show()

print("Hierarchical group sizes:")
display(pd.Series(ward_labels).value_counts().sort_index().to_frame("rows"))
ward_kmeans_ari = adjusted_rand_score(
    final_labels[sample_positions], ward_labels
)
print("ARI: Ward versus K-means on sampled rows:", f"{ward_kmeans_ari:.3f}")
print(
    "This sampled dendrogram is a teaching view, not full-data validation."
)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Can you identify the largest jumps in fusion height?
- Does the cut produce the requested number of groups?
- Are some groups very small?
- Does the K-means comparison use the same sampled rows?
- Why can two valid algorithms disagree?
- What would be required to generalize the sampled dendrogram to the full dataset?

# Part 7: Linkage Sensitivity

A single dendrogram can look authoritative. Comparing linkage methods makes the algorithmic choice visible. The goal is not to select whichever tree tells the most appealing story; it is to report whether the substantive conclusion survives reasonable alternatives.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 7: Compare Ward, Complete, and Average Linkage

        ```text
        Using X_hierarchical, selected_k, and the sampled K-means labels:

1. fit Ward, complete, and average linkage trees;
2. cut each tree into selected_k groups;
3. calculate silhouette score for each partition on the sampled rows;
4. calculate ARI against sampled K-means labels;
5. calculate pairwise ARI among the three hierarchical partitions;
6. show one comparison table and three compact dendrograms;
7. do not choose a winner using only one metric;
8. write a printed interpretation prompt about linkage sensitivity.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 7


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 7


In [ ]:
sampled_kmeans_labels = final_labels[sample_positions]
linkage_results = {}
comparison_rows = []

for method in ["ward", "complete", "average"]:
    tree = linkage(X_hierarchical, method=method, metric="euclidean")
    labels_method = fcluster(
        tree, t=selected_k, criterion="maxclust"
    )
    linkage_results[method] = {"tree": tree, "labels": labels_method}
    comparison_rows.append(
        {
            "linkage": method,
            "clusters_returned": int(np.unique(labels_method).size),
            "silhouette": silhouette_score(
                X_hierarchical, labels_method
            ),
            "ari_vs_kmeans": adjusted_rand_score(
                sampled_kmeans_labels, labels_method
            ),
        }
    )

linkage_comparison = pd.DataFrame(comparison_rows)
display(linkage_comparison)

methods = list(linkage_results)
pairwise_ari = pd.DataFrame(index=methods, columns=methods, dtype=float)
for method_a in methods:
    for method_b in methods:
        pairwise_ari.loc[method_a, method_b] = adjusted_rand_score(
            linkage_results[method_a]["labels"],
            linkage_results[method_b]["labels"],
        )
print("Pairwise ARI among hierarchical partitions:")
display(pairwise_ari)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for axis, method in zip(axes, methods):
    dendrogram(
        linkage_results[method]["tree"],
        ax=axis,
        no_labels=True,
        color_threshold=None,
    )
    axis.set_title(f"{method.title()} linkage")
    axis.set_xlabel("Sampled observations")
    axis.set_ylabel("Fusion height")
plt.tight_layout()
plt.show()

print(
    "Interpretation prompt: which profile conclusions remain stable "
    "when the definition of between-cluster distance changes?"
)


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 Look Back

1. What information does a dendrogram encode?
2. Why can leaf order be misleading?
3. How do single, complete, average, and Ward linkage differ?
4. Which linkage choices produced similar partitions?
5. Did hierarchical clustering agree with K-means?
6. Why is disagreement informative rather than automatically a failure?
7. How does sampling limit the dendrogram?
8. Which cluster profiles survive reasonable representation and algorithm changes?
9. What external evidence would support a meaningful interpretation?
10. Why should consequential decisions not depend on an unexplained cluster number?


# Sources and Course Resources

- James, Gareth, Daniela Witten, Trevor Hastie, and Robert Tibshirani. _An Introduction to Statistical Learning with Applications in Python_, Sections 12.1 and 12.4.
- Original credit-card behavior dataset: [Kaggle Credit Card Dataset for Clustering](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata).
- _The Ethical Algorithm_, Chapter 3, “Shopping with 300 Million Friends,” pages 116–123. **Background social-good reference; not assigned.**
- Earlier course notebook: `unsupervised-learning-tutorial.ipynb`.
